# 04 - RRAO Multi-Profile Comparison

Synthetic engineering and validation evidence, not final regulatory capital.

Use this notebook with the [RRAO package journey](../docs/PACKAGE_JOURNEY.md) and the executable [package demo](../examples/run_demo.py).

Raw inputs your upstream risk engine must emit: one RRAO positions table with deterministic position identifiers, legal-entity and desk lineage, gross effective notional, evidence type, product descriptors, and exclusion evidence fields. The machine-readable contract is [`frtb_rrao.positions.schema.json`](../../../docs/schemas/input_table/frtb_rrao.positions.schema.json); the package dataset contract is [`DATASET_CONTRACT.md`](../docs/DATASET_CONTRACT.md).

This notebook compares the RRAO capital result for the common position subset
under Basel MAR23 and US NPR 2.0, then isolates the positions that are specific
to the US NPR 2.0 scope.

## Key findings (spoiler)
- For the 14 included positions shared by both profiles, capital is **identical**:
  the risk weights (1% / 0.1%) are the same in Basel MAR23 and US NPR 2.0.
- The differences are entirely in **scope**: US NPR adds supervisor-directed
  inclusion, investment-fund treatment, and an expanded exclusion catalogue.
- Three of the six US NPR-only positions are exclusions; the other three are one
  supervisor-directed and two investment-fund positions — the NPR exclusion set
  is wider, giving filers more ways to zero out notional.

Regulatory anchors: Basel MAR23.2–MAR23.8; U.S. NPR 2.0 proposed `__.211(a)`–`__.211(c)`.  
Demonstration caution: outputs are synthetic fixture evidence.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from examples.rrao_fixture import (
    load_context,
    load_positions,
    PROFILE_US_NPR,
    PROFILE_BASEL,
    COMMON_POSITION_IDS,
    US_NPR_ONLY_POSITION_IDS,
)
from frtb_rrao import (
    calculate_rrao_capital,
    RraoClassification,
    RraoExclusionReason,
    RraoInputError,
    classify_rrao_position,
    get_rrao_rule_profile,
)
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Load the common subset (valid under both profiles)
common_positions = load_positions(position_ids=COMMON_POSITION_IDS)
npr_only_positions = load_positions(position_ids=US_NPR_ONLY_POSITION_IDS)
all_positions = load_positions()

ctx_npr = load_context(profile=PROFILE_US_NPR)
ctx_basel = load_context(profile=PROFILE_BASEL)

print(f"Common positions (Basel + US NPR): {len(common_positions)}")
print(f"US NPR-only positions: {len(npr_only_positions)}")


## Public API happy path

This notebook compares the same public capital entrypoint under the Basel MAR23 and US NPR 2.0 profile contexts.


In [ ]:
from frtb_rrao import calculate_rrao_capital

public_basel = calculate_rrao_capital(common_positions, context=ctx_basel)
public_npr = calculate_rrao_capital(common_positions, context=ctx_npr)
print(f"Basel common-subset RRAO : {public_basel.total_rrao:,.2f}")
print(f"US NPR common-subset RRAO: {public_npr.total_rrao:,.2f}")
print(f"Basel profile hash       : {public_basel.profile_hash}")
print(f"US NPR profile hash      : {public_npr.profile_hash}")


## Implementation details / math verification - Common Positions: Capital Under Both Profiles

Running the same 19 positions through both profiles shows that the capital
result is identical — because Basel MAR23 and US NPR 2.0 use the same risk
weights for the shared position types (1% EXOTIC, 0.1% OTHER_RESIDUAL_RISK).

The differences show only in the citation IDs: Basel cites MAR23 paragraphs;
NPR cites `__.211` subsections.

In [ ]:
result_basel = calculate_rrao_capital(common_positions, context=ctx_basel)
result_npr_common = calculate_rrao_capital(common_positions, context=ctx_npr)

print(f"Basel MAR23 — common subset RRAO:   ${result_basel.total_rrao:,.0f}")
print(f"US NPR 2.0 — common subset RRAO:    ${result_npr_common.total_rrao:,.0f}")
print()
match = result_basel.total_rrao == result_npr_common.total_rrao
print(f"Capital result identical: {'YES — risk weights and scope match' if match else 'NO — INVESTIGATE'}")

## Side-by-Side Line Comparison

For included positions, we show the add-on computed under each profile alongside
the cited paragraphs.  The add-on numbers are always the same; the citations differ.

In [ ]:
lines_basel = {l.position_id: l for l in result_basel.lines}
lines_npr = {l.position_id: l for l in result_npr_common.lines}

# Only look at included positions
shared_ids = sorted(set(lines_basel) & set(lines_npr))

rows = []
for pid in shared_ids:
    bl = lines_basel[pid]
    nl = lines_npr[pid]
    agree = "✓" if bl.add_on == nl.add_on else "✗"
    rows.append([
        pid,
        bl.classification.value,
        f"{bl.risk_weight:.1%}",
        f"${bl.add_on:,.0f}",
        f"${nl.add_on:,.0f}",
        agree,
        bl.citations[0] if bl.citations else "",
        nl.citations[0] if nl.citations else "",
    ])

display(Markdown(
    "| Position | Classification | Wt | Basel Add-on | NPR Add-on | Match | Basel Citation | NPR Citation |\n"
    "|---|---|---|---|---|---|---|---|\n"
    + "\n".join(f"| {' | '.join(r)} |" for r in rows)
))

## Profile Scope Comparison

This chart visualises where the profiles diverge.  The US NPR 2.0 scope
captures additional positions that Basel MAR23 does not define:
- **Supervisor-directed** inclusion (Basel has no equivalent)
- **Investment-fund** included-exposure treatment (Basel has no equivalent)
- Three additional **exclusion reasons** (deliverable hedge pair, government
  or GSE debt, fallback capital requirement)

In [ ]:
# Full US NPR run (all positions)
result_npr_full = calculate_rrao_capital(all_positions, context=ctx_npr)

# US NPR-only included lines
npr_only_incl = [l for l in result_npr_full.lines
                 if l.position_id in US_NPR_ONLY_POSITION_IDS]
npr_only_excl = [l for l in result_npr_full.excluded_lines
                 if l.position_id in US_NPR_ONLY_POSITION_IDS]

npr_only_addon = sum(l.add_on for l in npr_only_incl)
common_addon = result_npr_common.total_rrao

fig, ax = plt.subplots(figsize=(9, 4.5))

categories = ["Basel MAR23\nCommon Scope", "US NPR 2.0\nCommon Scope", "US NPR 2.0\nNPR-Only Scope", "US NPR 2.0\nFull Book"]
values = [
    result_basel.total_rrao,
    common_addon,
    npr_only_addon,
    result_npr_full.total_rrao,
]
colors = ["#4a8fcc", "#4a8fcc", "#e07030", "#7a5cc0"]
bars = ax.bar(categories, values, color=colors, alpha=0.86)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5000,
            f"${v:,.0f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_title("RRAO by profile scope — Basel MAR23 vs US NPR 2.0")
ax.set_ylabel("USD add-on")
plt.tight_layout()
plt.show()

print(f"Incremental NPR-only capital (supervisor + fund + NPR excl.): ${npr_only_addon:,.0f}")
print(f"NPR-only includes {len(npr_only_incl)} included + {len(npr_only_excl)} excluded positions")

## Exclusion Rule Coverage Comparison

The US NPR 2.0 exclusion catalogue is broader than Basel MAR23.  This table
maps each exclusion reason to the profile(s) that support it.  A wider exclusion
catalogue allows filers to remove more notional from the RRAO base, subject to
the evidential requirements of each exclusion path.

In [ ]:
from frtb_rrao.reference_data import exclusion_rules_for_profile

basel_excl = {r.exclusion_reason for r in exclusion_rules_for_profile(PROFILE_BASEL)}
npr_excl = {r.exclusion_reason for r in exclusion_rules_for_profile(PROFILE_US_NPR)}
all_excl = sorted(basel_excl | npr_excl, key=lambda r: r.value)

rows_excl = []
for reason in all_excl:
    in_basel = "✓" if reason in basel_excl else "—"
    in_npr = "✓" if reason in npr_excl else "—"
    note = "Basel + NPR" if reason in basel_excl and reason in npr_excl else (
        "NPR only" if reason in npr_excl else "Basel only"
    )
    rows_excl.append([reason.value, in_basel, in_npr, note])

display(Markdown(
    "| Exclusion Reason | Basel MAR23 | US NPR 2.0 | Profile Coverage |\n"
    "|---|---|---|---|\n"
    + "\n".join(f"| {' | '.join(r)} |" for r in rows_excl)
))

print(f"\nBasel MAR23 exclusion reasons:  {len(basel_excl)}")
print(f"US NPR 2.0 exclusion reasons:   {len(npr_excl)}")
print(f"NPR-only exclusion reasons:     {len(npr_excl - basel_excl)}")


## What Would Fail Under Basel MAR23

Running the US NPR-only positions through the Basel MAR23 profile illustrates
which features are profile-specific.  The model rejects these positions with
deterministic errors rather than silently computing a wrong result.

In [ ]:
print("Attempting to classify US NPR-only positions under Basel MAR23:")
print()
for pos in npr_only_positions:
    try:
        decision = classify_rrao_position(pos, profile=PROFILE_BASEL)
        print(f"  {pos.position_id}: OK (unexpected)")
    except RraoInputError as exc:
        print(f"  {pos.position_id}: [RraoInputError] {exc}")


## Profile Hashes

Each profile has a deterministic content hash derived from its citation map,
evidence rules, exclusion rules, risk-weight rules, and investment-fund rules.
A change to any profile component changes its hash — this makes it possible to
detect rule-table changes across releases.

In [ ]:
for profile in (PROFILE_BASEL, PROFILE_US_NPR):
    rp = get_rrao_rule_profile(profile)
    print(f"{profile.value}")
    print(f"  Regulator:              {rp.regulator}")
    print(f"  Version:                {rp.version}")
    print(f"  Content hash:           {rp.content_hash}")
    print(f"  Supported classif.:     {sorted(c.value for c in rp.supported_classifications)}")
    print(f"  Supported evidence:     {len(rp.supported_evidence_types)} types")
    print(f"  Supported exclusions:   {len(rp.supported_exclusions)} reasons")
    print()


## See also

- [RRAO package journey](../docs/PACKAGE_JOURNEY.md)
- [RRAO dataset contract](../docs/DATASET_CONTRACT.md)
- [Client integration guide](../../../docs/CLIENT_INTEGRATION.md)
- [SBM package journey](../../frtb-sbm/docs/PACKAGE_JOURNEY.md)
- [DRC package journey](../../frtb-drc/docs/PACKAGE_JOURNEY.md)
- [CVA package journey](../../frtb-cva/docs/PACKAGE_JOURNEY.md)
- [IMA package journey](../../frtb-ima/docs/PACKAGE_JOURNEY.md)
